In [3]:
import xarray as xr
import pandas as pd
from pathlib import Path

# --- MidCal bbox ---
lat_min, lat_max = 36.0, 37.5
lon_min, lon_max = -124.0, -122.0

out_csv = Path("../../1_DATA/processed/oisst_midcal_bbox_monthly.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

# Monthly OISST via OPeNDAP (no NCSS)
url = "https://psl.noaa.gov/thredds/dodsC/Datasets/noaa.oisst.v2.highres/sst.mon.mean.nc"
ds = xr.open_dataset(url)  # remote

# detect coord names
lat_name  = "lat" if "lat" in ds.coords else "latitude"
lon_name  = "lon" if "lon" in ds.coords else "longitude"
time_name = "time"

# lon handling (dataset often uses 0..360)
lon = ds[lon_name]
if float(lon.max()) > 180:
    lon_min_use = (lon_min + 360) % 360
    lon_max_use = (lon_max + 360) % 360
else:
    lon_min_use, lon_max_use = lon_min, lon_max

sst = ds["sst"]

# subset + spatial mean (monthly already)
sst_bbox = sst.sel({lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min_use, lon_max_use)})
if sst_bbox.sizes.get(lat_name, 0) == 0:  # descending lat safeguard
    sst_bbox = sst.sel({lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min_use, lon_max_use)})

sst_m = sst_bbox.mean(dim=[lat_name, lon_name], skipna=True).to_series()
sst_m.index = pd.to_datetime(sst_m.index)
sst_m = sst_m.sort_index()
sst_m.name = "sst"

# anomaly vs fixed baseline (1991–2020 if available)
baseline = sst_m.loc["1991-01-01":"2020-12-01"] if len(sst_m.loc["1991":"2020"]) else sst_m
clim = baseline.groupby(baseline.index.month).mean()
sst_anom_m = sst_m - sst_m.index.month.map(clim)

df_out = pd.DataFrame({"sst": sst_m, "sst_anom": sst_anom_m})
df_out.to_csv(out_csv, index=True)

print("Saved:", out_csv.resolve())
print("Range:", df_out.index.min(), "to", df_out.index.max(), "rows:", len(df_out))
print(df_out.head())

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/oisst_midcal_bbox_monthly.csv
Range: 1981-09-01 00:00:00 to 2026-01-01 00:00:00 rows: 533
                  sst  sst_anom
time                           
1981-09-01  13.837731 -1.739406
1981-10-01  14.205532 -0.829856
1981-11-01  14.518955  0.439978
1981-12-01  13.206291  0.191680
1982-01-01  11.977019 -0.539124


In [8]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("../../1_DATA/processed/midcal/oisst_midcal_bbox_monthly.csv")
kelp_path  = Path("../../1_DATA/processed/midcal/kelp_timeseries_midcal_bbox.csv")
out_path   = Path("../../1_DATA/processed/midcal/oisst_features_midcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

# load
sst_m = pd.read_csv(sst_m_path, parse_dates=["time"])
sst_m = sst_m.set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

# quarterly features from monthly
q = sst_m.resample("QS")  # quarter start buckets

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

# lag (1 quarter)
feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

# align to kelp quarters (using q-start)
feat = feat.reindex(kelp_qstart)
feat.index = kelp_times  # set index to kelp timestamps for easy join later

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/oisst_features_midcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   13.029227        0.523896       0.816057             1.801960
1984-05-15   12.322297       -0.563491      -0.464906             0.523896
1984-08-15   15.775856        0.665889       0.922719            -0.563491
1984-11-15   13.939277       -0.103716       0.176351             0.665889
1985-02-15   11.962078       -0.543253       0.049712            -0.103716
1985-05-15   12.687473       -0.198315       0.693891            -0.543253
1985-08-15   15.343442        0.233475       0.889905            -0.198315
1985-11-15   13.138452       -0.904540      -0.054317             0.233475
1986-02-15   13.422072        0.916741       1.323946            -0.904540
1986-05-15   13.062651        0.176864       0.979759             0.916741


In [9]:
import pandas as pd
from pathlib import Path

# --- MIDCAL paths ---
sst_m_path = Path("../../1_DATA/processed/midcal/oisst_midcal_bbox_monthly.csv")
kelp_path  = Path("../../1_DATA/processed/midcal/kelp_timeseries_midcal_bbox.csv")
out_path   = Path("../../1_DATA/processed/midcal/oisst_features_midcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

# --- load monthly SST (CSV saved by the OPeNDAP method) ---
sst_m = pd.read_csv(sst_m_path, parse_dates=[0])
sst_m = sst_m.rename(columns={sst_m.columns[0]: "time"}).set_index("time").sort_index()

# --- load kelp quarterly ---
df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

# --- quarterly features from monthly ---
q = sst_m.resample("QS")  # quarter start

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
    "sstanom_pos_sum": q["sst_anom"].apply(lambda x: x[x > 0].sum()),          # heat stress (cumulative)
    "sstanom_exceed_months": q["sst_anom"].apply(lambda x: (x > x.quantile(0.9)).sum()),  # duration proxy (within-quarter)
})

# --- lag features ---
feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)
feat["sstanom_pos_sum_lag1"] = feat["sstanom_pos_sum"].shift(1)
feat["sstanom_q_max_lag1"] = feat["sstanom_q_max"].shift(1)

# --- align to kelp quarters ---
feat = feat.reindex(kelp_qstart)
feat.index = kelp_times  # match kelp timestamps

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/oisst_features_midcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_pos_sum  \
1984-02-15   13.029227        0.523896       0.816057         1.571688   
1984-05-15   12.322297       -0.563491      -0.464906         0.000000   
1984-08-15   15.775856        0.665889       0.922719         1.997667   
1984-11-15   13.939277       -0.103716       0.176351         0.176351   
1985-02-15   11.962078       -0.543253       0.049712         0.049712   
1985-05-15   12.687473       -0.198315       0.693891         0.693891   
1985-08-15   15.343442        0.233475       0.889905         0.905934   
1985-11-15   13.138452       -0.904540      -0.054317         0.000000   
1986-02-15   13.422072        0.916741       1.323946         2.750223   
1986-05-15   13.062651        0.176864       0.979759         1.312201   

            sstanom_exceed_months  sstanom_q_mean_lag1  sstanom_pos_sum_lag1  \


In [4]:
import pandas as pd
from pathlib import Path

sst_m_path = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/oisst_midcal_bbox_monthly.csv")
kelp_path  = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/kelp_timeseries_midcal_bbox.csv")
out_path   = Path("/Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/oisst_features_midcal_quarterly.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

sst_m = pd.read_csv(sst_m_path, parse_dates=["time"]).set_index("time").sort_index()

df_kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp_times = pd.DatetimeIndex(df_kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

q = sst_m.resample("QS")

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "sstanom_q_mean": q["sst_anom"].mean(),
    "sstanom_q_max": q["sst_anom"].max(),
})

feat["sstanom_q_mean_lag1"] = feat["sstanom_q_mean"].shift(1)

feat = feat.reindex(kelp_qstart)
feat.index = kelp_times

feat.to_csv(out_path, index=True)
print("Saved:", out_path.resolve())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/oisst_features_midcal_quarterly.csv
            sst_q_mean  sstanom_q_mean  sstanom_q_max  sstanom_q_mean_lag1
1984-02-15   13.029227        0.523896       0.816057             1.801960
1984-05-15   12.322297       -0.563491      -0.464906             0.523896
1984-08-15   15.775856        0.665889       0.922719            -0.563491
1984-11-15   13.939277       -0.103716       0.176351             0.665889
1985-02-15   11.962078       -0.543253       0.049712            -0.103716
1985-05-15   12.687473       -0.198315       0.693891            -0.543253
1985-08-15   15.343442        0.233475       0.889905            -0.198315
1985-11-15   13.138452       -0.904540      -0.054317             0.233475
1986-02-15   13.422072        0.916741       1.323946            -0.904540
1986-05-15   13.062651        0.176864       0.979759             0.916741


In [5]:
# graph rq
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# load monthly NorCal SST (made earlier)
sst_path = Path("../../1_DATA/processed/midcal/oisst_midcal_bbox_monthly.csv")
d = pd.read_csv(sst_path, parse_dates=["time"]).set_index("time").sort_index()

fig_dir = Path("../../5_FIGURES/midcal")
fig_dir.mkdir(parents=True, exist_ok=True)
outpath = fig_dir / "midcal_sst_monthly.png"

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(d.index, d["sst"])
ax.set_title("NorCal OISST SST (monthly mean)")
ax.set_xlabel("Year")
ax.set_ylabel("SST (°C)")
fig.tight_layout()

fig.savefig(outpath, dpi=200, bbox_inches="tight")
plt.close(fig)

print("saved:", outpath.resolve(), "bytes:", outpath.stat().st_size)

saved: /Users/tonylin/Documents/kelp_project/5_FIGURES/midcal/midcal_sst_monthly.png bytes: 200459
